# Support Vector Regression — fitting a tube, not a line

> Tutorial pair for [`svr.py`](svr.py).

## 1. Intuition
Ordinary least squares punishes *every* deviation, however tiny. SVR instead
draws a tube of half-width $\varepsilon$ around the function and says: **if a
point lands inside the tube, its error is zero — don't move for it.** Only points
that poke *outside* the tube exert force on the model. The result is a sparse,
robust regressor that ignores small noise and is steered only by the points that
matter (the support vectors). Nonlinearity comes from a kernel / feature map,
exactly as in classification SVMs.

## 2. Concept (the slide)
- **$\varepsilon$-insensitive tube:** errors $\le\varepsilon$ cost nothing;
  beyond that the cost grows linearly (an L1-like, robust penalty).
- **Objective:** $\tfrac12\lVert w\rVert^2 + C\sum_i L_\varepsilon(y_i-f(x_i))$.
  $\lVert w\rVert^2$ keeps the function flat/simple; $C$ weights fit vs flatness.
- **Support vectors:** only points on or outside the tube ($|y_i-f(x_i)|\ge\varepsilon$).
- **Kernels:** replace $x$ by features $\phi(x)$; here we use **random Fourier
  features** to approximate the RBF kernel cheaply and explicitly.

## 3. Math derivation — the $\varepsilon$-insensitive loss & its sub-gradient

### The loss
$$L_\varepsilon(r)=\max\big(0,\;|r|-\varepsilon\big),\qquad r=y-f(x).$$
A flat-bottomed valley: zero on $[-\varepsilon,\varepsilon]$, then slope $\pm1$.

### Primal objective
$$\min_{w,b}\;J(w,b)=\tfrac12\lVert w\rVert^2
 + C\sum_{i=1}^n \max\!\big(0,\;|y_i-(w^\top\phi(x_i)+b)|-\varepsilon\big).$$

### Sub-gradient
$L_\varepsilon$ is non-differentiable at the kinks, so we use a **sub-gradient**.
With residual $r_i=y_i-f(x_i)$,
$$\frac{\partial L_\varepsilon}{\partial r_i}=
\begin{cases}
0 & |r_i|\le\varepsilon \quad(\text{inside the tube})\\
-\operatorname{sign}(r_i) & |r_i|>\varepsilon \quad(\text{outside})
\end{cases}$$
Chaining through $f=w^\top\phi(x)+b$ (note $\partial r_i/\partial w=-\phi(x_i)$),
let $s_i=-\operatorname{sign}(r_i)\,\mathbf 1[|r_i|>\varepsilon]$. Then
$$\boxed{\;\nabla_w J = w + C\sum_i s_i\,\phi(x_i),\qquad
 \nabla_b J = C\sum_i s_i.\;}$$
Gradient descent on this is what `SVRNumPy` runs; PyTorch reproduces the same
sub-gradient via autograd on `clamp(|r|-eps, min=0)`.

### Dual / kernel view (for context)
Introducing multipliers $\alpha_i,\alpha_i^*$ for the two sides of the tube gives
the dual
$$\max -\tfrac12\sum_{i,j}(\alpha_i-\alpha_i^*)(\alpha_j-\alpha_j^*)K(x_i,x_j)
 -\varepsilon\sum_i(\alpha_i+\alpha_i^*)+\sum_i y_i(\alpha_i-\alpha_i^*)$$
subject to $\sum_i(\alpha_i-\alpha_i^*)=0$, $0\le\alpha_i,\alpha_i^*\le C$, with
$f(x)=\sum_i(\alpha_i-\alpha_i^*)K(x_i,x)+b$. KKT forces $\alpha_i=\alpha_i^*=0$
for points strictly inside the tube — that is the sparsity. We instead optimize
the **primal** directly and get RBF nonlinearity from random Fourier features:
$$\phi(x)=\sqrt{\tfrac{2}{D}}\cos(Wx+b_{\text{rf}}),\quad W\sim\mathcal N(0,2\gamma I)
 \;\Rightarrow\; \langle\phi(x),\phi(x')\rangle\approx e^{-\gamma\lVert x-x'\rVert^2}.$$

## 4. NumPy implementation — primal SVR by sub-gradient descent

In [ ]:
# ===== actual implementation from svr.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def make_rff(n_features, n_rff=200, gamma=0.5, seed=SEED):
    """Return a map phi(X)->(n, n_rff) with E[<phi(x),phi(x')>] = exp(-gamma||x-x'||^2)."""
    rng = np.random.default_rng(seed)
    # Bochner's theorem: RBF kernel <-> Gaussian spectral density.
    W = rng.normal(scale=np.sqrt(2 * gamma), size=(n_features, n_rff))
    b = rng.uniform(0, 2 * np.pi, size=n_rff)

    def phi(X):
        X = np.asarray(X, float)
        return np.sqrt(2.0 / n_rff) * np.cos(X @ W + b)
    return phi

class SVRNumPy:
    r"""
    Primal epsilon-insensitive SVR.

    Objective (epsilon-insensitive loss + L2 margin term):
        min_{w,b}  (1/2)||w||^2 + C sum_i L_eps( y_i - (w^T phi(x_i)+b) )
    where the eps-insensitive loss is
        L_eps(r) = max(0, |r| - eps)        (free inside the tube |r| <= eps).

    Sub-gradient wrt the residual r_i = y_i - f(x_i):
        dL/dr = 0                if |r_i| <= eps      (inside tube, no push)
              = -sign(r_i)       if |r_i| >  eps      (pull f toward y_i)
    Chain through f = w^T phi + b:
        grad_w = w + C * sum_i s_i * phi(x_i),   grad_b = C * sum_i s_i,
        where s_i = -sign(r_i) * 1[|r_i|>eps]  (i.e. s_i = +1 above tube, -1 below).
    """

    def __init__(self, C=1.0, epsilon=0.1, kernel="linear", n_rff=200, gamma=0.5,
                 lr=0.01, n_iters=500, seed=SEED):
        self.C = C
        self.epsilon = epsilon
        self.kernel = kernel
        self.n_rff = n_rff
        self.gamma = gamma
        self.lr = lr
        self.n_iters = n_iters
        self.seed = seed

    def _phi(self, X):
        return self._map(X) if self.kernel == "rbf" else np.asarray(X, float)

    def fit(self, X, y):
        X = np.asarray(X, float)
        y = np.asarray(y, float)
        if self.kernel == "rbf":
            self._map = make_rff(X.shape[1], self.n_rff, self.gamma, self.seed)
        Z = self._phi(X)                          # feature representation
        n, d = Z.shape
        w = np.zeros(d)
        b = 0.0
        self.history = []
        for _ in range(self.n_iters):
            f = Z @ w + b                         # current predictions
            r = y - f                             # residuals
            outside = np.abs(r) > self.epsilon    # points outside the tube
            # s_i = +1 if prediction is below y (r>0), -1 if above; 0 inside tube
            s = np.where(outside, -np.sign(r), 0.0)
            grad_w = w + self.C * (Z.T @ s)       # d/dw of 1/2||w||^2 + C*loss
            grad_b = self.C * s.sum()
            w -= self.lr * grad_w / n
            b -= self.lr * grad_b / n
            loss = 0.5 * w @ w + self.C * np.maximum(0, np.abs(r) - self.epsilon).sum()
            self.history.append(loss)
        self.w, self.b = w, b
        self.n_support_ = int(outside.sum())      # points outside the tube
        return self

    def predict(self, X):
        return self._phi(X) @ self.w + self.b

## 5. PyTorch implementation — same objective via autograd

In [ ]:
# ===== actual implementation from svr.py =====
import torch

import torch.nn as nn

def get_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

class SVRTorch(nn.Module):
    r"""
    Primal SVR in PyTorch. Forward gives f(x)=w^T phi(x)+b; the loss is
        (1/2)||w||^2 + C * sum_i max(0, |y_i - f(x_i)| - eps).
    Autograd supplies the sub-gradient of the eps-insensitive loss automatically.
    RBF nonlinearity uses explicit random Fourier features (same as NumPy).
    """

    def __init__(self, n_features, C=1.0, epsilon=0.1, kernel="linear",
                 n_rff=200, gamma=0.5, seed=SEED):
        super().__init__()
        self.C = C
        self.epsilon = epsilon
        self.kernel = kernel
        if kernel == "rbf":
            g = torch.Generator().manual_seed(seed)
            self.register_buffer("Wrf", torch.randn(n_features, n_rff, generator=g) * (2 * gamma) ** 0.5)
            self.register_buffer("brf", torch.rand(n_rff, generator=g) * 2 * np.pi)
            self.n_rff = n_rff
            self.linear = nn.Linear(n_rff, 1)
        else:
            self.linear = nn.Linear(n_features, 1)

    def _phi(self, X):
        if self.kernel == "rbf":
            return (2.0 / self.n_rff) ** 0.5 * torch.cos(X @ self.Wrf + self.brf)
        return X

    def forward(self, X):
        return self.linear(self._phi(X)).squeeze(-1)

    def fit(self, X, y, lr=0.05, n_iters=500):
        dev = get_device(); self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        y = torch.as_tensor(y, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(n_iters):
            opt.zero_grad()
            r = (y - self(X)).abs()
            eps_loss = torch.clamp(r - self.epsilon, min=0).sum()
            reg = 0.5 * (self.linear.weight ** 2).sum()
            loss = reg + self.C * eps_loss
            loss.backward(); opt.step()
            self.history.append(loss.item())
        return self

    @torch.no_grad()
    def predict(self, X):
        dev = next(self.parameters()).device
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        return self(X).cpu().numpy()

def _rmse(a, b):
    return float(np.sqrt(np.mean((a - b) ** 2)))

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    # Tiny problems: a single BLAS thread avoids CPU thread-oversubscription
    # overhead that otherwise dominates these many small matmuls.
    torch.set_num_threads(1)
    rng = np.random.default_rng(SEED)

    # --- linear target with noise ---
    n = 200
    Xl = rng.uniform(-3, 3, size=(n, 1))
    yl = (1.7 * Xl[:, 0] - 0.5 + rng.normal(scale=0.3, size=n))
    Xtr, ytr, Xte, yte = Xl[:160], yl[:160], Xl[160:], yl[160:]

    print("=== Linear SVR ===")
    lin = SVRNumPy(C=1.0, epsilon=0.1, kernel="linear", lr=0.05, n_iters=600).fit(Xtr, ytr)
    print(f"  NumPy  test RMSE = {_rmse(lin.predict(Xte), yte):.3f}  "
          f"(pts outside tube = {lin.n_support_}/{len(Xtr)})")
    tl = SVRTorch(1, C=1.0, epsilon=0.1, kernel="linear").fit(Xtr, ytr, lr=0.05, n_iters=600)
    print(f"  Torch  test RMSE = {_rmse(tl.predict(Xte), yte):.3f}")

    # --- nonlinear target: needs the RBF feature map ---
    Xn = rng.uniform(-3, 3, size=(n, 1))
    yn = np.sin(Xn[:, 0]) + 0.3 * Xn[:, 0] + rng.normal(scale=0.1, size=n)
    Xn = (Xn - Xn.mean(0)) / Xn.std(0)
    Xtr, ytr, Xte, yte = Xn[:160], yn[:160], Xn[160:], yn[160:]

    print("\n=== Nonlinear target (y = sin x + 0.3x) ===")
    lin_n = SVRNumPy(C=1.0, epsilon=0.05, kernel="linear", lr=0.05, n_iters=600).fit(Xtr, ytr)
    print(f"  NumPy linear SVR test RMSE = {_rmse(lin_n.predict(Xte), yte):.3f}  <- underfits")
    rbf_n = SVRNumPy(C=2.0, epsilon=0.05, kernel="rbf", n_rff=200, gamma=1.0,
                     lr=0.1, n_iters=1000).fit(Xtr, ytr)
    print(f"  NumPy RBF SVR    test RMSE = {_rmse(rbf_n.predict(Xte), yte):.3f}")
    rbf_t = SVRTorch(1, C=2.0, epsilon=0.05, kernel="rbf", n_rff=200, gamma=1.0).fit(
        Xtr, ytr, lr=0.1, n_iters=600)
    print(f"  Torch RBF SVR    test RMSE = {_rmse(rbf_t.predict(Xte), yte):.3f}")

## 6. Train — linear vs RBF SVR, NumPy vs Torch

In [ ]:
demo()

## 7. Visualization — the $\varepsilon$-tube and the RBF fit

Left: a linear SVR with its $\pm\varepsilon$ tube (points inside cost nothing).
Right: on a nonlinear target the linear fit underfits while the RBF feature map
tracks the curve.

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
import svr as M

rng = np.random.default_rng(0)

# Left: linear SVR + epsilon tube
Xl = rng.uniform(-3, 3, size=(150, 1))
yl = 1.7 * Xl[:, 0] - 0.5 + rng.normal(scale=0.3, size=150)
lin = M.SVRNumPy(C=1.0, epsilon=0.4, kernel="linear", lr=0.05, n_iters=600).fit(Xl, yl)
xs = np.linspace(-3, 3, 200)[:, None]
f = lin.predict(xs)

# Right: nonlinear target, linear vs RBF
Xn = rng.uniform(-3, 3, size=(200, 1))
yn = np.sin(Xn[:, 0]) + 0.3 * Xn[:, 0] + rng.normal(scale=0.1, size=200)
Xn = (Xn - Xn.mean(0)) / Xn.std(0)
lin_n = M.SVRNumPy(C=1.0, epsilon=0.05, kernel="linear", lr=0.05, n_iters=600).fit(Xn, yn)
rbf_n = M.SVRNumPy(C=2.0, epsilon=0.05, kernel="rbf", n_rff=200, gamma=1.0, lr=0.1, n_iters=1000).fit(Xn, yn)
order = np.argsort(Xn[:, 0]); xo = Xn[order]

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
ax[0].scatter(Xl[:, 0], yl, s=12, alpha=.5)
ax[0].plot(xs[:, 0], f, "b-", label="SVR fit")
ax[0].plot(xs[:, 0], f + lin.epsilon, "b--", alpha=.6, label="$\\pm\\varepsilon$ tube")
ax[0].plot(xs[:, 0], f - lin.epsilon, "b--", alpha=.6)
ax[0].set_title("Linear SVR with $\\varepsilon$-tube"); ax[0].legend()

ax[1].scatter(Xn[:, 0], yn, s=12, alpha=.4)
ax[1].plot(xo[:, 0], lin_n.predict(xo), "g-", label="linear (underfits)")
ax[1].plot(xo[:, 0], rbf_n.predict(xo), "r-", label="RBF features")
ax[1].set_title("Nonlinear target: linear vs RBF SVR"); ax[1].legend()
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- **$\varepsilon$** sets how much error you call "noise" (a dead-zone); larger
  $\varepsilon$ → sparser model, smoother, more bias. **$C$** trades flatness
  against fitting the out-of-tube points.
- SVR's L1-style penalty outside the tube makes it **robust to outliers**
  compared with squared-error regression.
- For nonlinear targets you need a kernel / feature map; the **random Fourier
  features** here approximate RBF in $O(nD)$ instead of forming an $n\times n$
  Gram matrix, so it scales.
- Standardize inputs (and often the target): the tube width $\varepsilon$ and RBF
  $\gamma$ are both scale-dependent.
- The sub-gradient is zero inside the tube — if $\varepsilon$ is too large the
  model gets *no* signal and stalls flat.